# HI 5304 — APIs & JSON in Python (Jupyter Notebook)

*Created: 2026-01-25*

This notebook teaches the essentials of working with **APIs** and **JSON** in Python using **Jupyter** (ideal for GitHub Codespaces). It is **read-only** and uses public endpoints.


## Learning goals
- Make a GET request to a public API
- Understand JSON → Python dictionaries/lists
- Handle parameters, headers, and common errors
- Normalize JSON into a `pandas` DataFrame
- Practice with short exercises


## 0. Setup
Most Codespaces images already include `requests`. If you need it, run the install cell once.


In [ ]:
# Optional: run once if needed
# %pip install requests pandas

import requests
import json
import pandas as pd


## 1. What is an API?
**API** (Application Programming Interface) endpoints let software exchange data.

Most modern APIs return data as **JSON** (JavaScript Object Notation), which maps cleanly to Python:
- JSON object → Python `dict`
- JSON array → Python `list`


## 2. Your first API call (GitHub)
We'll call GitHub’s public API to fetch metadata about the class repo.


In [ ]:
import requests
import json
import pandas as pd


In [ ]:
url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks"

r = requests.get(url, headers={"User-Agent": "hi5304"})
r.raise_for_status()  # raises an error for 4xx/5xx responses

data = r.json()  # JSON -> Python dict
type(data), list(data.keys())[:10]


### Inspect a few fields


In [ ]:
data["full_name"], data["open_issues_count"], data["stargazers_count"], data["updated_at"]


### Pretty-print JSON (first 1200 characters)
This helps you *see* nesting and field names.


In [ ]:
print(json.dumps(data, indent=2)[:1200])


## 3. Nested JSON
Many APIs (including FHIR) use nested objects.
Here, `owner` is a nested dictionary.


In [ ]:
type(data["owner"]), data["owner"]["login"], data["owner"]["html_url"]


### Safer access with `.get()`
APIs can change. `.get()` avoids `KeyError` if a field is missing.


In [ ]:
license_info = data.get("license")  # could be None
language = data.get("language", "Not specified")

license_info, language


## 4. Common request components
- **URL**: the endpoint
- **Query parameters**: e.g., `?per_page=10`
- **Headers**: metadata (often auth + content type)
- **Timeouts**: prevent hanging forever


### Example: parameters (`params=`)
Fetch the 10 most recent issues (if any).


In [ ]:
issues_url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks/issues"

r = requests.get(
    issues_url,
    params={"per_page": 10, "state": "open"},
    headers={"User-Agent": "hi5304"},
    timeout=10
)
r.raise_for_status()
issues = r.json()

type(issues), len(issues)


### Normalize a list of JSON objects into a DataFrame
If there are no open issues, the DataFrame will just be empty.


In [ ]:
df_issues = pd.json_normalize(issues)
df_issues.head()


## 5. Error handling patterns (teachable + practical)
Students should learn how to interpret errors and respond safely.


In [ ]:
def safe_get(url: str, *, params=None, headers=None, timeout=10):
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=timeout)
        resp.raise_for_status()
        return resp
    except requests.exceptions.Timeout:
        print("⏱️ Timeout: the server took too long to respond.")
    except requests.exceptions.HTTPError as e:
        print(f"🚫 HTTP error: {e} | status={getattr(resp, 'status_code', None)}")
        try:
            print("Response body (first 300 chars):", resp.text[:300])
        except Exception:
            pass
    except requests.exceptions.RequestException as e:
        print("❗ Network/request error:", e)

resp = safe_get("https://api.github.com/rate_limit", headers={"User-Agent":"hi5304"})
rate_data = resp.json() if resp else None
rate_data and rate_data["rate"]


## 6. Rate limits (important for real-world APIs)
Many APIs limit requests per hour/day. GitHub exposes rate limit information.


In [ ]:
if rate_data:
    core = rate_data["resources"]["core"]
    print("Limit:", core["limit"])
    print("Remaining:", core["remaining"])
    print("Resets (unix time):", core["reset"])


## 7. API keys & secrets (DO NOT commit keys)
For course work, the safest pattern is:
- Store keys in **Codespaces Secrets** or an untracked `.env`
- Read them via environment variables

Example (won't print the full key):


In [ ]:
import os

# Example only: set a Codespaces secret like OPENWEATHER_API_KEY (do not hardcode keys)
api_key = os.getenv("OPENWEATHER_API_KEY")

if api_key:
    print("✅ API key loaded (first 4 chars):", api_key[:4] + "...")
else:
    print("ℹ️ No API key found. That's okay for this notebook's public endpoints.")


## 8. Mini-project: pull commit activity and summarize
We'll fetch recent commits and build a tidy table.


In [ ]:
commits_url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks/commits"

resp = safe_get(commits_url, params={"per_page": 15}, headers={"User-Agent":"hi5304"})
commits = resp.json() if resp else []

rows = []
for c in commits:
    rows.append({
        "sha": c.get("sha", "")[:10],
        "author": (c.get("commit", {}).get("author", {}) or {}).get("name"),
        "date": (c.get("commit", {}).get("author", {}) or {}).get("date"),
        "message": (c.get("commit", {}) or {}).get("message", "").splitlines()[0][:80],
    })

df_commits = pd.DataFrame(rows)
df_commits.head(10)


### Quick summary


In [ ]:
df_commits["author"].value_counts(dropna=False).head(10)


## 9. Exercises (for students)
1) Change `per_page` from 15 to 5 in the commits request. What changes?
2) Add a column called `message_len` with the length of each commit message.
3) Using the `issues` endpoint, request `state=all` and find how many issues are returned.
4) (Challenge) Create a function that takes an owner + repo name and returns a DataFrame summary.


In [ ]:
# Exercise 2 starter: add message_len
df_commits["message_len"] = df_commits["message"].fillna("").str.len()
df_commits[["sha","author","date","message_len","message"]].head(10)


## 10. Extension: swap in a health-focused API
Once this pattern is comfortable, we can repeat the same workflow with public health APIs like:
- CDC public datasets (Socrata)
- OpenFDA
- U.S. Census

Tell me which one you want, and I’ll provide a drop-in section.


Example: OpenFDA Drug Adverse Events API
What this API represents (context for students)

OpenFDA provides access to FDA Adverse Event Reporting System (FAERS) data:

Reports of side effects, medication errors, and adverse reactions

Used in pharmacovigilance and post-market surveillance

Real-world, messy, high-volume public health data

🔗 Conceptually similar to:

Disease surveillance feeds

Safety signal monitoring systems

Real-world evidence pipelines

Step 1: Define the endpoint

This query asks:

“Give me the 10 most recent adverse event reports involving metformin.”

In [ ]:
import requests
import json
import pandas as pd

url = "https://api.fda.gov/drug/event.json"

params = {
    "search": "patient.drug.medicinalproduct:metformin",
    "limit": 10
}

r = requests.get(url, params=params, timeout=10)
r.raise_for_status()

data = r.json()
type(data), data.keys()


Step 2: Inspect the structure

In [ ]:
print(json.dumps(data, indent=2)[:1500])


Step 3: Extract the results

In [ ]:
events = data["results"]
len(events), type(events)


Step 4: Normalize JSON into a DataFrame

In [ ]:
rows = []

for e in events:
    patient = e.get("patient", {})
    drugs = patient.get("drug", [])
    reactions = patient.get("reaction", [])

    rows.append({
        "safety_report_id": e.get("safetyreportid"),
        "received_date": e.get("receivedate"),
        "serious": e.get("serious"),
        "drug_count": len(drugs),
        "reaction_count": len(reactions),
        "reactions": "; ".join(
            [r.get("reactionmeddrapt", "") for r in reactions[:3]]
        )
    })

df = pd.DataFrame(rows)
df


CDC (Socrata): Population health / surveillance-style indicators

Example dataset: CDC U.S. Chronic Disease Indicators (Socrata dataset ID: hksd-2xuw).

In [ ]:
import requests
import pandas as pd

# Socrata SODA endpoint pattern:
# https://data.cdc.gov/resource/<dataset_id>.json
url = "https://data.cdc.gov/resource/hksd-2xuw.json"

params = {
    "$select": "yearstart,locationdesc,topic,question,datavalueunit,datavalue",
    "$where": "yearstart >= 2020 AND locationdesc = 'Texas'",
    "$limit": 25
}

r = requests.get(url, params=params, timeout=20)
r.raise_for_status()
rows = r.json()

df = pd.DataFrame(rows)
df.head(10)


U.S. Census API: SDOH / demographics (ACS)

Example: ACS 5-year (total population variable B01001_001E) using the Census “examples” endpoint for 2022.

In [ ]:
import requests
import pandas as pd
import os

# Census API base (ACS 5-year, 2022 example)
base = "https://api.census.gov/data/2022/acs/acs5"

# Optional: Census API key (recommended). If you don't have one, this may still work but can be rate-limited.
key = os.getenv("CENSUS_API_KEY")

params = {
    "get": "NAME,B01001_001E",   # NAME + total population estimate
    "for": "state:*"            # all states
}
if key:
    params["key"] = key

r = requests.get(base, params=params, timeout=20)
r.raise_for_status()
data = r.json()

# First row is headers
cols = data[0]
rows = data[1:]

df = pd.DataFrame(rows, columns=cols)
df["B01001_001E"] = pd.to_numeric(df["B01001_001E"], errors="coerce")
df.sort_values("B01001_001E", ascending=False).head(10)


CMS Data API: Utilization / cost / quality measures (data.cms.gov)

CMS provides dataset-specific API docs pages that include the dataset UUID(s) you query via /dataset/{uuid}/data.

Example dataset: Medicare Inpatient Hospitals – by Provider
Use the 2023 version UUID shown on the CMS API docs page: ad17abc4-0e93-4828-bba5-e8f39ddafbdb.

In [ ]:
import requests
import pandas as pd

uuid_2023 = "ad17abc4-0e93-4828-bba5-e8f39ddafbdb"
url = f"https://data.cms.gov/data-api/v1/dataset/{uuid_2023}/data"

params = {
    "limit": 25,
    # You can filter with "query" for simple text matching.
    # For structured filtering/column selection, check the dataset's api-docs page.
    "query": "Texas"
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
payload = r.json()

# CMS returns objects; often the records are in payload (list) or payload['data'] depending on endpoint/version
rows = payload if isinstance(payload, list) else payload.get("data", payload)

df = pd.DataFrame(rows)
df.head(10)
